# Partie 2/3 - Validation independante (Personne 2)
Controle qualite, puis calcul des 3 indicateurs (revenus par ville, duree moyenne, top 5 stations) **de 3 facons** : Python pur (reference), RDD et DataFrame. Les trois doivent concorder.

In [ ]:
import os, sys, time
from pathlib import Path
sys.path.append(os.environ.get("PEDALI_DATA_MODULE", "../data"))
from pedali_schema import *
from pyspark.sql import SparkSession, functions as F

# Jupyter (docker) : SPARK_MASTER=spark://spark-master:7077 ; en local : local[*]
spark = (SparkSession.builder.appName("pedali-p2")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.shuffle.partitions", "8")   # donnees petites : 200 partitions (defaut) = surcout inutile
         .getOrCreate())
DATA = Path(os.environ.get("PEDALI_DATA", "../data")).resolve()
print(spark.version, spark.sparkContext.master, spark.sparkContext.defaultParallelism)

## 1. Lecture avec schema explicite et controle qualite

In [ ]:
trips = read_trips(spark, str(DATA / "trips_history.jsonl"))
stations = read_stations(spark, str(DATA / "stations.json"))
trips.printSchema()
for k, v in quality_report(trips, stations).items():
    print(f"{k:32s} {v}")

In [ ]:
clean = clean_trips(trips, stations).filter("status = 'completed'").cache()
print("trajets termines apres nettoyage :", clean.count())

## 2. Reference en Python pur (meme logique que `data/validate_data.py`)

In [ ]:
import json
from collections import Counter, defaultdict
st = {s["station_id"] for s in json.loads((DATA / "stations.json").read_text(encoding="utf8"))}
seen, ref_rows = set(), []
for line in (DATA / "trips_history.jsonl").read_text(encoding="utf8").splitlines():
    t = json.loads(line)
    if t["trip_id"] in seen or not t["user_id"] or t["start_station_id"] not in st: continue
    if t["duration_min"] is not None and t["duration_min"] < 0: continue
    seen.add(t["trip_id"])
    if t["status"] == "completed": ref_rows.append(t)
ref_rev = defaultdict(float)
for t in ref_rows: ref_rev[t["city"]] += t["price_mad"]
ref_rev = {k: round(v, 2) for k, v in ref_rev.items()}
ref_avg = round(sum(t["duration_min"] for t in ref_rows) / len(ref_rows), 2)
ref_top = Counter(t["start_station_id"] for t in ref_rows).most_common(5)
print(ref_rev, ref_avg, ref_top, sep="\n")

## 3. Version RDD

In [ ]:
rdd = clean.rdd
rdd_rev = {k: round(v, 2) for k, v in rdd.map(lambda r: (r.city, r.price_mad)).reduceByKey(lambda a, b: a + b).collect()}
n, tot = rdd.map(lambda r: (1, r.duration_min)).reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))
rdd_avg = round(tot / n, 2)
rdd_top = rdd.map(lambda r: (r.start_station_id, 1)).reduceByKey(lambda a, b: a + b).takeOrdered(5, key=lambda x: -x[1])
print(rdd_rev, rdd_avg, rdd_top, sep="\n")

## 4. Version DataFrame

In [ ]:
df_rev = {r.city: round(r.revenu, 2) for r in clean.groupBy("city").agg(F.sum("price_mad").alias("revenu")).collect()}
df_avg = round(clean.agg(F.avg("duration_min")).first()[0], 2)
df_top = [(r.start_station_id, r["count"]) for r in clean.groupBy("start_station_id").count().orderBy(F.desc("count")).limit(5).collect()]
print(df_rev, df_avg, df_top, sep="\n")

## 5. Verification : les 3 resultats concordent

In [ ]:
assert ref_rev == rdd_rev == df_rev, "revenus par ville differents"
assert ref_avg == rdd_avg == df_avg, "duree moyenne differente"
assert ref_top == rdd_top == df_top, "top 5 differents"
print("OK : Python pur == RDD == DataFrame")